# 01 Dense Embedding Architectures: InfoNCE, Matryoshka (MRL) & ColBERT

Implementing PyTorch InfoNCE Contrastive Loss, Matryoshka Representation Learning (MRL) nested dimension slicing, and ColBERT token-level late interaction.

## Setup: Repository-Level Dotenv Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load repo-level .env file
load_dotenv()

print("Phase 3 Vector DB Environment Loaded.")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))

## Technique 1: PyTorch InfoNCE Contrastive Loss Function

In [ ]:
# PyTorch InfoNCE Contrastive Loss Implementation
import torch
import torch.nn as nn
import torch.nn.functional as F

class InfoNCELoss(nn.Module):
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, query_embeddings: torch.Tensor, key_embeddings: torch.Tensor) -> torch.Tensor:
        # Normalize embeddings to unit vectors
        q_norm = F.normalize(query_embeddings, p=2, dim=-1)
        k_norm = F.normalize(key_embeddings, p=2, dim=-1)
        
        # Cosine similarity matrix (Batch x Batch)
        logits = torch.matmul(q_norm, k_norm.T) / self.temperature
        
        # Ground truth labels: diagonal elements are positive pairs
        labels = torch.arange(logits.size(0), device=logits.device)
        loss = F.cross_entropy(logits, labels)
        return loss

# Execution
q_emb = torch.randn(4, 128)
k_emb = q_emb + torch.randn(4, 128) * 0.1 # Positive pairs close to query
loss_fn = InfoNCELoss(temperature=0.07)
loss_val = loss_fn(q_emb, k_emb)
print(f"InfoNCE Contrastive Loss Value: {loss_val.item():.4f}")

## Technique 2: Matryoshka (MRL) Nested Embedding Slicing

In [ ]:
# Matryoshka Representation Learning (MRL) Nested Slicing
class MatryoshkaEmbeddingSlicer:
    def __init__(self, full_vector: np.ndarray):
        self.full_vector = full_vector

    def slice_dimensions(self, target_dim: int) -> np.ndarray:
        sliced = self.full_vector[:target_dim]
        # Renormalize sliced vector to unit length
        return sliced / np.linalg.norm(sliced)

# Execution
full_emb = np.random.randn(1024)
slicer = MatryoshkaEmbeddingSlicer(full_emb)

emb_128 = slicer.slice_dimensions(128)
emb_512 = slicer.slice_dimensions(512)

print(f"Full Vector Shape: {full_emb.shape}")
print(f"Sliced 128d Norm: {np.linalg.norm(emb_128):.2f} | Shape: {emb_128.shape}")
print(f"Sliced 512d Norm: {np.linalg.norm(emb_512):.2f} | Shape: {emb_512.shape}")